# 🐍 Day 2: Thread Pools

- ThreadPoolExecutor
- submit, as_completed
- map
- concurrent.futures

## Why Thread Pools?

Reuse threads instead of creating new ones for each task. `concurrent.futures` provides a clean API.

## ThreadPoolExecutor Basics

Create a pool, submit tasks, get results via Future objects.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def task(n):
    time.sleep(0.5)
    return n * 2

with ThreadPoolExecutor(max_workers=3) as executor:
    future = executor.submit(task, 5)
    result = future.result()
    print(result)

## submit Multiple Tasks

Submit many tasks and collect results.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def task(n):
    time.sleep(0.2)
    return n ** 2

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(task, i) for i in range(8)]
    results = [f.result() for f in futures]

print(results)

## as_completed

Process results as they complete, not in submission order.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import random

def task(n):
    delay = random.uniform(0.1, 0.5)
    time.sleep(delay)
    return (n, delay)

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(task, i): i for i in range(6)}
    for future in as_completed(futures):
        n, delay = future.result()
        print(f"Task {n} completed in {delay:.2f}s")

## map

Like built-in map but parallel. Results in same order as inputs.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def fetch(url_id):
    time.sleep(0.2)
    return f"data-{url_id}"

urls = [1, 2, 3, 4, 5]
with ThreadPoolExecutor(max_workers=3) as executor:
    results = list(executor.map(fetch, urls))

print(results)

## map with timeout

`map` supports `timeout`; raises `TimeoutError` if not all complete in time.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def slow_task(n):
    time.sleep(n * 0.5)
    return n

with ThreadPoolExecutor(max_workers=2) as executor:
    try:
        results = list(executor.map(slow_task, [1, 2, 3], timeout=1.0))
    except TimeoutError:
        print("Timeout - some tasks did not complete")

## Handling Exceptions

Exceptions in tasks surface when calling `result()`.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def may_fail(n):
    if n == 2:
        raise ValueError("n is 2")
    return n

with ThreadPoolExecutor(max_workers=2) as executor:
    futures = [executor.submit(may_fail, i) for i in range(4)]
    for f in futures:
        try:
            print(f.result())
        except ValueError as e:
            print(f"Error: {e}")

## Real Example: Parallel HTTP Requests

Fetch multiple URLs concurrently (simulated).

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def fetch_url(url):
    time.sleep(0.3)  # Simulate network
    return f"{url}: OK"

urls = ["https://a.com", "https://b.com", "https://c.com", "https://d.com"]
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(fetch_url, u): u for u in urls}
    for future in as_completed(futures):
        url = futures[future]
        print(future.result())
print(f"Done in {time.perf_counter() - start:.2f}s")

## Common Mistakes

- **Blocking on result() too early**: Use as_completed for streaming
- **Too many workers**: More threads != faster (GIL, context switch)
- **Not using with**: Executor.shutdown() is automatic with with
- **map with different-length iterables**: Stops at shortest

## Exercises

In [ ]:
# Exercise 1: Use ThreadPoolExecutor to compute squares of 1-10 in parallel.
# YOUR CODE HERE

In [ ]:
# Exercise 2: Use as_completed to print results as they arrive (with varying delays).
# YOUR CODE HERE

In [ ]:
# Exercise 3: Use executor.map with chunksize=2 for a list of 10 items.
# YOUR CODE HERE

In [ ]:
# Exercise 4: Submit 5 tasks, handle one that raises; collect successful results.
# YOUR CODE HERE

In [ ]:
# Exercise 5: Compare sequential vs parallel time for 8 I/O-bound tasks (sleep 0.2s each).
# YOUR CODE HERE

In [ ]:
# Exercise 6: Use future.done() to check completion without blocking.
# YOUR CODE HERE

## Key Takeaways

- ThreadPoolExecutor: submit, map, as_completed
- as_completed: process results as they finish
- map: same order as input
- Use with for automatic shutdown

## 🔜 Next: Day 3 - Multiprocessing

Tomorrow: Process, Pool, Queue, Pipe!